# YT SEO Insights Generator
### AI-Powered YouTube SEO Analysis

> **Stack:** Groq llama-3.3-70b-versatile | YouTube scraping | MLflow | Streamlit  
> **Colab-ready:** All secrets from Colab Secrets (lock icon in sidebar)

---

## Cell 1 - Project Configuration
All paths, audio settings, and MLflow experiment name.

In [1]:
import os

# ── Hardcoded Colab paths ─────────────────────────────────────────────────────
AUDIO_DIR   = '/content/audio'      # <- upload your audio tracks here
IMAGE_DIR   = '/content/images'     # <- upload your cover image here
OUTPUT_DIR  = '/content/output'     # <- all generated files land here

# Allowed audio file extensions for the mixtape builder
ALLOWED_AUDIO_EXT = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg')

# ── Mixtape settings ──────────────────────────────────────────────────────────
TRANSITION_MS  = 6000           # crossfade duration in milliseconds
OUTPUT_MP3     = 'mixtape.mp3'
OUTPUT_MP4     = 'mixtape_vid.mp4'
MIXTAPE_NAME   = 'Afro House Summer Mix'
GENRE          = 'Afro House'
COVER_IMAGE    = 'cover.jpg'    # filename inside IMAGE_DIR

# ── MLflow experiment ─────────────────────────────────────────────────────────
MLFLOW_EXPERIMENT = 'youtube-mixtape-automation'

# ── Create required runtime directories ──────────────────────────────────────
for d in [AUDIO_DIR, IMAGE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Config loaded.')
print(f'  Audio dir  : {AUDIO_DIR}')
print(f'  Image dir  : {IMAGE_DIR}')
print(f'  Output dir : {OUTPUT_DIR}')
print(f'  Transition : {TRANSITION_MS} ms')


Config loaded.
  Audio dir  : /content/audio
  Image dir  : /content/images
  Output dir : /content/output
  Transition : 6000 ms


## Cell 2 - Install Dependencies

In [2]:
# Install all runtime dependencies.
# streamlit     : web UI framework
# groq          : Groq API client (replaces openai)
# python-dotenv : loads .env files (parity with local dev)
# requests      : HTTP scraping of YouTube pages
# mlflow        : experiment tracking and artifact logging
# pyngrok       : exposes local Streamlit port to a public URL

%pip install -q streamlit groq python-dotenv requests mlflow pyngrok
print('All dependencies installed.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/83

## Cell 3 - Load API Keys from Colab Secrets
Keys are read from the **Colab Secret Store** (lock icon in left sidebar).  
Add `GROQ_API_KEY` there before running this cell.

In [3]:
from google.colab import userdata  # Colab built-in secret manager

# ── Read Groq API key from Colab Secrets ──────────────────────────────────────
# Sidebar -> Secrets (lock icon) -> + Add new secret
# Name : GROQ_API_KEY   Value: gsk_...
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError(
        'GROQ_API_KEY not found in Colab Secrets. '
        'Please add it via the lock icon in the left sidebar.'
    )

# Inject into environment so the Groq SDK picks it up automatically
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('GROQ_API_KEY loaded from Colab Secrets.')


GROQ_API_KEY loaded from Colab Secrets.


## Cell 4 - Logger (src/common/logger.py)
File-based logger that writes timestamped logs to `logs/`.  
Every module obtains its logger via `get_logger(__name__)`.

In [4]:
import os
import logging
from datetime import datetime

# ── Logging directory ─────────────────────────────────────────────────────────
LOGS_DIR = 'logs'
os.makedirs(LOGS_DIR, exist_ok=True)

# ── Log file: one file per session named with current timestamp ───────────────
LOG_FILE = os.path.join(
    LOGS_DIR,
    f"log_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.log"
)

# ── Root logger configuration ─────────────────────────────────────────────────
logging.basicConfig(
    filename=LOG_FILE,
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

def get_logger(name: str) -> logging.Logger:
    """Return a named logger that inherits the root configuration."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    return logger

print(f'Logger initialised. Log file -> {LOG_FILE}')


Logger initialised. Log file -> logs/log_2026-03-24_04-54-46.log


## Cell 5 - Custom Exception (src/common/custom_exception.py)
Enriches every exception with the **file name** and **line number** where it originated, making debugging easier in deep call stacks.

In [5]:
import sys

class CustomException(Exception):
    """
    Project-wide exception class.

    Enriches any exception with:
      - original message
      - root-cause exception (if provided)
      - source file name and line number at the raise point
    """

    def __init__(self, message: str, error_detail=None):
        # Build the enriched message and store it
        self.error_message = self._build_message(message, error_detail)
        super().__init__(self.error_message)

    @staticmethod
    def _build_message(message: str, error_detail) -> str:
        """Construct a detailed error string using the active traceback."""
        _, _, exc_tb = sys.exc_info()

        # Guard: exc_tb is None when called outside an except block
        if exc_tb is not None:
            file_name   = exc_tb.tb_frame.f_code.co_filename
            line_number = exc_tb.tb_lineno
        else:
            file_name   = 'unknown'
            line_number = 'unknown'

        return (
            f"{message} | "
            f"Error: {error_detail} | "
            f"File: {file_name} | "
            f"Line: {line_number}"
        )

    def __str__(self) -> str:
        return self.error_message

print('CustomException class defined.')


CustomException class defined.


## Cell 6 - Video Extractor (src/utils/video_extractor.py)
Scrapes YouTube metadata (title, author, views, duration, thumbnail) directly from the page HTML. No YouTube Data API key required.

Supported URL formats:
- `https://www.youtube.com/watch?v=VIDEO_ID`
- `https://youtu.be/VIDEO_ID`
- `https://www.youtube.com/shorts/VIDEO_ID`

In [6]:
import re
import requests

class VideoExtractor:
    """
    Extracts YouTube video metadata by scraping the watch-page HTML.

    No YouTube Data API key needed; metadata is parsed from the
    embedded JSON-LD and Open Graph tags on every watch page.
    """

    def __init__(self):
        self.logger = get_logger(__name__)
        self.logger.info('VideoExtractor initialised.')

    # ── Private helpers ───────────────────────────────────────────────────────

    def _extract_video_id(self, url: str) -> str:
        """
        Parse a YouTube URL and return the 11-character video ID.

        Supports standard watch links, youtu.be, and YouTube Shorts.

        Raises:
            CustomException: if URL is empty, not a string, or
                             has no recognisable video ID.
        """
        if not url or not isinstance(url, str):
            raise CustomException('URL is empty or not a string.')

        # Three regex patterns covering the most common YouTube URL formats
        patterns = [
            r'(?:v=|\/)([0-9A-Za-z_-]{11})',           # ?v= and /VIDEO_ID paths
            r'youtu\.be\/([0-9A-Za-z_-]{11})',          # shortened links
            r'youtube\.com\/shorts\/([0-9A-Za-z_-]{11})'  # Shorts
        ]

        for pattern in patterns:
            match = re.search(pattern, url)
            if match:
                video_id = match.group(1)
                self.logger.info(f'Extracted video ID: {video_id}')
                return video_id

        raise CustomException('Could not extract a valid 11-char video ID from the URL.')

    def _fetch_youtube_metadata(self, video_id: str) -> dict:
        """
        Fetch the YouTube watch page and extract metadata via regex.

        Returns dict with keys:
            title, thumbnail_url, duration (int seconds),
            views (int), author, platform, video_id.

        Raises:
            CustomException: on HTTP error or parse failure.
        """
        self.logger.info(f'Fetching metadata for video_id={video_id}')

        watch_url = f'https://www.youtube.com/watch?v={video_id}'
        headers   = {'User-Agent': 'Mozilla/5.0'}  # mimic a real browser

        response = requests.get(watch_url, headers=headers, timeout=10)

        if response.status_code != 200:
            self.logger.error(f'HTTP {response.status_code} for {video_id}')
            raise CustomException(f'HTTP request failed (status {response.status_code}).')

        html = response.text

        def _re(pattern: str, default: str = '') -> str:
            """Return first capture group from regex or default."""
            m = re.search(pattern, html)
            return m.group(1) if m else default

        # ── Extract individual metadata fields ────────────────────────────────
        title     = _re(r'<meta property="og:title" content="([^"]+)"', 'Untitled Video')
        thumbnail = f'https://img.youtube.com/vi/{video_id}/hqdefault.jpg'
        duration  = int(_re(r'"lengthSeconds":"(\d+)"', '400'))  # seconds
        views     = int(_re(r'"viewCount":"(\d+)"',     '100'))
        author    = _re(r'"author":"([^"]+)"',           'Unknown Creator')

        metadata = {
            'title'        : title,
            'thumbnail_url': thumbnail,
            'duration'     : duration,
            'views'        : views,
            'author'       : author,
            'platform'     : 'YouTube',
            'video_id'     : video_id,
        }

        self.logger.info('Metadata extracted successfully.')
        return metadata

    # ── Public API ────────────────────────────────────────────────────────────

    def get_video_metadata(self, url: str) -> dict:
        """Parse URL, fetch and return metadata dict."""
        try:
            self.logger.info(f'Processing URL: {url}')
            video_id = self._extract_video_id(url)
            return self._fetch_youtube_metadata(video_id)
        except Exception as e:
            self.logger.error(f'Error processing URL: {e}')
            raise CustomException('Error while processing the YouTube URL.', e)


# ── Module-level convenience function ─────────────────────────────────────────

def get_video_metadata(url: str) -> dict:
    """
    Shortcut: create VideoExtractor and return metadata for url.

    Example:
        meta = get_video_metadata('https://www.youtube.com/watch?v=dQw4w9WgXcQ')
    """
    return VideoExtractor().get_video_metadata(url)

print('VideoExtractor class defined.')


VideoExtractor class defined.


## Cell 7 - SEO Engine (src/core/seo_engine.py)
Sends video metadata to **Groq (llama-3.3-70b-versatile)** and parses structured JSON into:

| Key | Content |
|---|---|
| `tags` | 35 SEO hashtags |
| `audience` | Target audience paragraph |
| `timestamps` | Auto-generated chapter markers |
| `flaws` | 2-3 SEO issues with fixes |

In [7]:
import json
from groq import Groq

class SEOEngine:
    """
    Generates YouTube SEO insights from video metadata using Groq.

    Model used: llama-3.3-70b-versatile (fast, free-tier friendly).

    Workflow:
        1. _build_prompt()    - construct the structured JSON prompt
        2. Groq API call      - send prompt, receive raw text
        3. _parse_json()      - extract valid JSON from the response
        4. _validate_output() - assert all required keys are present
    """

    # Groq model to use — swap here if you want a different model
    MODEL = 'llama-3.3-70b-versatile'

    def __init__(self):
        self.logger = get_logger(__name__)
        self.logger.info('SEOEngine initialised.')

        # Guard: ensure API key is present before creating the client
        if not os.environ.get('GROQ_API_KEY'):
            raise CustomException(
                'GROQ_API_KEY not found. Run Cell 3 (Secrets) first.'
            )

        try:
            # Groq SDK picks up GROQ_API_KEY from the environment automatically
            self.client = Groq(api_key=os.environ['GROQ_API_KEY'])
            self.logger.info('Groq client connected.')
        except Exception as e:
            self.logger.error('Failed to initialise Groq client.')
            raise CustomException('Failed to initialise Groq client.', e)

    # ── Private helpers ───────────────────────────────────────────────────────

    def _build_prompt(self, metadata: dict) -> str:
        """
        Construct the prompt from video metadata.

        Timestamp count is scaled by duration:
            num_timestamps = clamp(duration_min / 2, 5, 15)

        Args:
            metadata: dict returned by VideoExtractor.

        Returns:
            Prompt string ready for the model.
        """
        try:
            title    = metadata['title']
            duration = metadata['duration']    # seconds
            platform = metadata['platform']

            minutes        = duration // 60
            # Clamp timestamp count between 5 and 15
            num_timestamps = min(15, max(5, int(minutes / 2)))

            prompt = (
                'You MUST respond with VALID JSON ONLY. '
                'No preamble, no markdown fences.\n\n'
                f'Video details:\n'
                f'  Title    : "{title}"\n'
                f'  Platform : {platform}\n'
                f'  Duration : {duration} seconds\n\n'
                'Return JSON exactly in this structure:\n\n'
                '{\n'
                '  "tags": ["tag1", ..., "tag35"],\n'
                '  "audience": "One paragraph describing the ideal target audience.",\n'
                '  "timestamps": [\n'
                '    {"time": "00:00", "description": "Intro"},\n'
                '    ...\n'
                '  ],\n'
                '  "flaws": [\n'
                '    {\n'
                '      "issue"        : "Specific SEO problem found in this video.",\n'
                '      "why_it_hurts" : "Why this lowers ranking or watch-time.",\n'
                '      "fix"          : "Clear, actionable improvement step."\n'
                '    },\n'
                '    ...\n'
                '  ]\n'
                '}\n\n'
                'Hard rules:\n'
                '- Produce EXACTLY 35 tags.\n'
                f'- Produce EXACTLY {num_timestamps} timestamps spread across the full duration.\n'
                '- Produce 2-3 flaws.\n'
                '- All content must be in English.\n'
            )
            return prompt

        except Exception as e:
            self.logger.error('Error building prompt.')
            raise CustomException('Error building prompt.', e)

    def _parse_json(self, raw_output: str) -> dict:
        """
        Parse the model's raw text into a Python dict.

        Strategy 1: direct json.loads().
        Strategy 2 (fallback): slice out the first complete {...} block,
        handling cases where the model adds markdown fences.

        Raises:
            CustomException: if neither strategy succeeds.
        """
        try:
            return json.loads(raw_output)
        except Exception:
            try:
                # Slice out the outermost JSON object
                start = raw_output.find('{')
                end   = raw_output.rfind('}') + 1
                return json.loads(raw_output[start:end])
            except Exception as e:
                raise CustomException('Failed to parse JSON from model output.', e)

    def _validate_output(self, data: dict) -> None:
        """Assert all four required keys are present in the parsed output."""
        for key in ['tags', 'audience', 'timestamps', 'flaws']:
            if key not in data:
                self.logger.error(f"Model output missing key: '{key}'")
                raise CustomException(f"Model output missing required key: '{key}'")

    # ── Public API ────────────────────────────────────────────────────────────

    def generate(self, video_metadata: dict) -> dict:
        """
        Full pipeline: prompt -> Groq API call -> parse -> validate -> return.

        Args:
            video_metadata: dict from VideoExtractor.get_video_metadata().

        Returns:
            dict with keys: tags, audience, timestamps, flaws.
        """
        try:
            self.logger.info('Starting SEO insights generation.')

            prompt = self._build_prompt(video_metadata)

            # ── Groq inference call ───────────────────────────────────────────
            # The Groq SDK mirrors the OpenAI chat completions interface,
            # so only the client class and model name differ from OpenAI.
            response = self.client.chat.completions.create(
                model=self.MODEL,
                temperature=0.4,   # low = deterministic, structured output
                messages=[
                    {'role': 'system', 'content': 'Return ONLY valid JSON. No extra text.'},
                    {'role': 'user',   'content': prompt}
                ]
            )

            raw = response.choices[0].message.content.strip()
            self.logger.info('Raw model output received from Groq.')

            data = self._parse_json(raw)
            self._validate_output(data)

            self.logger.info('SEO insights generated successfully.')
            return data

        except Exception as e:
            self.logger.error(f'SEO generation failed: {e}')
            raise CustomException('Error during SEO insights generation.', e)

print('SEOEngine class defined (Groq backend).')


SEOEngine class defined (Groq backend).


## Cell 8 - MLflow Experiment Tracking
Logs each run as an MLflow experiment:
- **Params:** video title, duration, platform
- **Metrics:** tag count, timestamp count, flaw count
- **Artifact:** full `seo_results.json`

In [8]:
import mlflow

# ── Point MLflow at a local directory inside OUTPUT_DIR ───────────────────────
# All run data (params, metrics, artifacts) will be stored there.
mlflow.set_tracking_uri(f'file://{OUTPUT_DIR}/mlruns')
mlflow.set_experiment(MLFLOW_EXPERIMENT)

def log_run_to_mlflow(metadata: dict, seo_data: dict) -> None:
    """
    Log one SEO generation run to MLflow.

    Logs:
        params   - video-level identifiers (title, platform, duration)
        metrics  - output counts (tags, timestamps, flaws)
        artifact - full seo_data dict as seo_results.json
    """
    with mlflow.start_run():
        # ── Parameters ────────────────────────────────────────────────────────
        mlflow.log_param('video_title',   metadata.get('title',    'unknown'))
        mlflow.log_param('platform',      metadata.get('platform', 'YouTube'))
        mlflow.log_param('duration_sec',  metadata.get('duration', 0))
        mlflow.log_param('author',        metadata.get('author',   'unknown'))

        # ── Metrics ───────────────────────────────────────────────────────────
        mlflow.log_metric('tag_count',       len(seo_data.get('tags',       [])))
        mlflow.log_metric('timestamp_count', len(seo_data.get('timestamps', [])))
        mlflow.log_metric('flaw_count',      len(seo_data.get('flaws',      [])))

        # ── Artifact: full JSON result file ───────────────────────────────────
        artifact_path = os.path.join(OUTPUT_DIR, 'seo_results.json')
        with open(artifact_path, 'w') as f:
            json.dump(seo_data, f, indent=2)
        mlflow.log_artifact(artifact_path)

    print(f'MLflow run logged. Artifact -> {artifact_path}')

print(f"MLflow configured. Experiment: '{MLFLOW_EXPERIMENT}'")


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/24 04:55:57 INFO mlflow.tracking.fluent: Experiment with name 'youtube-mixtape-automation' does not exist. Creating a new experiment.


MLflow configured. Experiment: 'youtube-mixtape-automation'


## Cell 9 - Run the Full Pipeline
Replace `YOUTUBE_URL` with any public YouTube link, then run this cell to:
1. Scrape video metadata
2. Generate SEO insights via GPT-4o-mini
3. Log the run to MLflow
4. Display all results inline

In [9]:
# ── Input: replace with any public YouTube URL ───────────────────────────────
YOUTUBE_URL = 'https://www.youtube.com/watch?v=dQw4w9WgXcQ'   # <- replace me

# ── Step 1: Extract video metadata ───────────────────────────────────────────
print('Fetching video metadata...')
metadata = get_video_metadata(YOUTUBE_URL)

print(f"  Title    : {metadata['title']}")
print(f"  Author   : {metadata['author']}")
print(f"  Views    : {metadata['views']:,}")
print(f"  Duration : {metadata['duration'] // 60} min {metadata['duration'] % 60} sec")

# ── Step 2: Generate SEO insights via Groq (llama-3.3-70b-versatile) ──────────
print('\nGenerating SEO insights...')
engine   = SEOEngine()
seo_data = engine.generate(metadata)

# ── Step 3: Log run to MLflow ─────────────────────────────────────────────────
print('\nLogging run to MLflow...')
log_run_to_mlflow(metadata, seo_data)

# ── Step 4: Display all results inline ───────────────────────────────────────
sep = '=' * 60

print(f'\n{sep}')
print('SEO-FRIENDLY TAGS (35)')
print(sep)
print('  '.join(f'#{t}' for t in seo_data['tags']))

print(f'\n{sep}')
print('TARGET AUDIENCE ANALYSIS')
print(sep)
print(seo_data['audience'])

print(f'\n{sep}')
print('AI-GENERATED TIMESTAMPS')
print(sep)
for ts in seo_data['timestamps']:
    print(f"  {ts['time']}  -  {ts['description']}")

print(f'\n{sep}')
print('FLAWS AND IMPROVEMENT SUGGESTIONS')
print(sep)
for i, flaw in enumerate(seo_data['flaws'], 1):
    print(f"\n  [{i}] Issue        : {flaw['issue']}")
    print(f"       Why it hurts : {flaw['why_it_hurts']}")
    print(f"       Fix          : {flaw['fix']}")

print('\nPipeline complete.')


INFO:__main__:VideoExtractor initialised.
INFO:__main__:Processing URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ
INFO:__main__:Extracted video ID: dQw4w9WgXcQ
INFO:__main__:Fetching metadata for video_id=dQw4w9WgXcQ


Fetching video metadata...


INFO:__main__:Metadata extracted successfully.
INFO:__main__:SEOEngine initialised.


  Title    : Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)
  Author   : Rick Astley
  Views    : 1,754,025,895
  Duration : 3 min 33 sec

Generating SEO insights...


INFO:__main__:Groq client connected.
INFO:__main__:Starting SEO insights generation.
INFO:__main__:Raw model output received from Groq.
INFO:__main__:SEO insights generated successfully.



Logging run to MLflow...
MLflow run logged. Artifact -> /content/output/seo_results.json

SEO-FRIENDLY TAGS (35)
#Rick Astley  #Never Gonna Give You Up  #Official Video  #4K Remaster  #80s Music  #Pop Music  #Music Video  #YouTube  #Rickroll  #Classic Song  #Nostalgia  #Throwback  #Music Legend  #Iconic Song  #Singer  #Songwriter  #Entertainer  #Music Icon  #80s Icon  #Pop Icon  #Legendary Song  #Timeless Music  #Eternal Song  #Music History  #YouTube Classic  #Viral Video  #Internet Meme  #Meme Song  #Funny Video  #Comedy Music  #Parody Song  #Cover Song  #Remastered Music  #High Quality Video  #4K Video  #Music Streaming

TARGET AUDIENCE ANALYSIS
The ideal target audience for this video is anyone who grew up in the 80s and 90s and is nostalgic for the music of that era, as well as younger viewers who are familiar with the song through its widespread use as a meme and are looking to experience the original version. This audience likely includes fans of classic pop music, retro enthus

## Cell 10 - Save Outputs and Checkpoint
Saves four artifact files to `OUTPUT_DIR`:
- `seo_results.json` — full structured output (also logged by MLflow)
- `seo_tags.txt` — hashtag list ready to paste into YouTube
- `seo_timestamps.txt` — chapter list for the video description
- `seo_flaws.txt` — flaw report

In [10]:
# ── Save full JSON results ────────────────────────────────────────────────────
results_json_path = os.path.join(OUTPUT_DIR, 'seo_results.json')
with open(results_json_path, 'w') as f:
    json.dump(seo_data, f, indent=2)
print(f'Saved full JSON    -> {results_json_path}')

# ── Save hashtag list (copy-paste ready for YouTube description) ──────────────
tags_txt_path = os.path.join(OUTPUT_DIR, 'seo_tags.txt')
with open(tags_txt_path, 'w') as f:
    f.write(' '.join(f'#{t}' for t in seo_data['tags']))
print(f'Saved tags list    -> {tags_txt_path}')

# ── Save timestamp list (copy-paste ready for YouTube description) ────────────
timestamps_txt_path = os.path.join(OUTPUT_DIR, 'seo_timestamps.txt')
with open(timestamps_txt_path, 'w') as f:
    for ts in seo_data['timestamps']:
        f.write(f"{ts['time']} {ts['description']}\n")
print(f'Saved timestamps   -> {timestamps_txt_path}')

# ── Save flaws / improvement report ──────────────────────────────────────────
flaws_txt_path = os.path.join(OUTPUT_DIR, 'seo_flaws.txt')
with open(flaws_txt_path, 'w') as f:
    for i, flaw in enumerate(seo_data['flaws'], 1):
        f.write(f"[{i}] Issue        : {flaw['issue']}\n")
        f.write(f"     Why it hurts : {flaw['why_it_hurts']}\n")
        f.write(f"     Fix          : {flaw['fix']}\n\n")
print(f'Saved flaws report -> {flaws_txt_path}')

print(f'\nAll outputs saved to: {OUTPUT_DIR}')


Saved full JSON    -> /content/output/seo_results.json
Saved tags list    -> /content/output/seo_tags.txt
Saved timestamps   -> /content/output/seo_timestamps.txt
Saved flaws report -> /content/output/seo_flaws.txt

All outputs saved to: /content/output


## Cell 11 - Launch Streamlit App (Optional)
Writes `app.py` and exposes it via **ngrok** so you can use the full web UI
from any browser while the Colab session is active.

**Prerequisite:** add `NGROK_AUTHTOKEN` to Colab Secrets (free at ngrok.com).

In [22]:
import subprocess
import time
from google.colab import userdata

APP_CODE = r'''
import os
import re
import sys
import json
import logging
import requests
from datetime import datetime
from dotenv import load_dotenv
import streamlit as st
from groq import Groq

load_dotenv()

# Logger
os.makedirs("logs", exist_ok=True)
LOG_FILE = os.path.join("logs", f"app_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.log")
logging.basicConfig(filename=LOG_FILE,
                    format="%(asctime)s - %(levelname)s - %(message)s",
                    level=logging.INFO)
def get_logger(name):
    lg = logging.getLogger(name)
    lg.setLevel(logging.INFO)
    return lg

logger = get_logger(__name__)

# CustomException
class CustomException(Exception):
    def __init__(self, message, error_detail=None):
        _, _, tb = sys.exc_info()
        file_name   = tb.tb_frame.f_code.co_filename if tb else "unknown"
        line_number = tb.tb_lineno if tb else "unknown"
        self.error_message = (f"{message} | Error: {error_detail} | "
                              f"File: {file_name} | Line: {line_number}")
        super().__init__(self.error_message)
    def __str__(self):
        return self.error_message

# VideoExtractor
def get_video_metadata(url):
    if not url or not isinstance(url, str):
        raise CustomException("URL is empty or invalid.")
    patterns = [
        r"(?:v=|\/)([0-9A-Za-z_-]{11})",
        r"youtu\.be\/([0-9A-Za-z_-]{11})",
        r"youtube\.com\/shorts\/([0-9A-Za-z_-]{11})"
    ]
    video_id = None
    for p in patterns:
        m = re.search(p, url)
        if m:
            video_id = m.group(1)
            break
    if not video_id:
        raise CustomException("Could not extract a valid video ID from the URL.")

    watch_url = f"https://www.youtube.com/watch?v={video_id}"
    headers   = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(watch_url, headers=headers, timeout=10)
    if resp.status_code != 200:
        raise CustomException(f"HTTP {resp.status_code} fetching video page.")
    html = resp.text

    def _re(pattern, default=""):
        m = re.search(pattern, html)
        return m.group(1) if m else default

    return {
        "title"        : _re(r'<meta property="og:title" content="([^"]+)"', "Untitled"),
        "thumbnail_url": f"https://img.youtube.com/vi/{video_id}/hqdefault.jpg",
        "duration"     : int(_re(r'"lengthSeconds":"(\d+)"', "400")),
        "views"        : int(_re(r'"viewCount":"(\d+)"',     "100")),
        "author"       : _re(r'"author":"([^"]+)"',           "Unknown"),
        "platform"     : "YouTube",
        "video_id"     : video_id,
    }

# SEOEngine
class SEOEngine:
    MODEL = "llama-3.3-70b-versatile"

    def __init__(self):
        self.logger = get_logger(__name__)
        api_key = os.environ.get("GROQ_API_KEY", "")
        if not api_key:
            raise CustomException("GROQ_API_KEY not set. Add it in the sidebar.")
        self.client = Groq(api_key=api_key)

    def _build_prompt(self, metadata):
        title    = metadata["title"]
        duration = metadata["duration"]
        platform = metadata["platform"]
        n_ts     = min(15, max(5, int((duration // 60) / 2)))
        return (
            "You MUST respond with VALID JSON ONLY. No preamble, no markdown fences.\n\n"
            f'Video details:\n  Title: "{title}"\n  Platform: {platform}\n  Duration: {duration}s\n\n'
            'Return JSON exactly:\n{\n'
            '  "tags": ["tag1",...,"tag35"],\n'
            '  "audience": "paragraph",\n'
            '  "timestamps": [{"time":"00:00","description":"Intro"},...],\n'
            '  "flaws": [{"issue":"...","why_it_hurts":"...","fix":"..."}]\n}\n\n'
            f'Rules: EXACTLY 35 tags, EXACTLY {n_ts} timestamps, 2-3 flaws, English only.'
        )

    def _parse_json(self, raw):
        try:
            return json.loads(raw)
        except Exception:
            start = raw.find("{")
            end   = raw.rfind("}") + 1
            return json.loads(raw[start:end])

    def generate(self, metadata):
        prompt   = self._build_prompt(metadata)
        response = self.client.chat.completions.create(
            model=self.MODEL,
            temperature=0.4,
            messages=[
                {"role": "system", "content": "Return ONLY valid JSON. No extra text."},
                {"role": "user",   "content": prompt}
            ]
        )
        raw  = response.choices[0].message.content.strip()
        data = self._parse_json(raw)
        for key in ["tags", "audience", "timestamps", "flaws"]:
            if key not in data:
                raise CustomException(f"Model output missing key: {key}")
        return data

# Streamlit UI
st.set_page_config(page_title="YT SEO Insights", layout="wide")
st.title("AI YouTube SEO Insights Generator")
st.write("AI-generated Tags | Audience | Timestamps | Flaws")

with st.sidebar:
    st.header("API Settings")
    api_key = st.text_input("Groq API Key", type="password")
    if api_key:
        os.environ["GROQ_API_KEY"] = api_key
        logger.info("API key set from sidebar.")

st.divider()
url = st.text_input("Enter YouTube URL",
                     placeholder="https://www.youtube.com/watch?v=...")

if "seo_data" not in st.session_state:
    st.session_state.seo_data = None

if url:
    try:
        metadata = get_video_metadata(url)
        title    = metadata.get("title",         "")
        author   = metadata.get("author",        "")
        views    = metadata.get("views",          0)
        duration = metadata.get("duration",       0)
        thumb    = metadata.get("thumbnail_url", "")

        st.subheader("Video Details")
        st.write(f"**Title:** {title}")
        st.write(f"**Creator:** {author}")
        st.write(f"**Views:** {views:,}")
        st.write(f"**Duration:** {duration // 60} min")
        st.image(thumb, width=400)

        if st.button("Generate SEO Insights"):
            if not os.getenv("GROQ_API_KEY"):
                st.error("Add your Groq API key in the sidebar first.")
            else:
                with st.spinner("Analysing with AI..."):
                    seo = SEOEngine()
                    st.session_state.seo_data = seo.generate(metadata)
    except Exception as e:
        logger.error(f"Error: {e}")
        st.error(f"Error: {e}")

data = st.session_state.seo_data
if data:
    st.subheader("SEO-Friendly Tags")
    cols = st.columns(3)
    for i, tag in enumerate(data.get("tags", [])):
        with cols[i % 3]:
            st.write(f"#{tag}")
    if st.button("Copy Tags"):
        st.code(" ".join(f"#{t}" for t in data.get("tags", [])))

    st.divider()
    st.subheader("Target Audience")
    st.write(data.get("audience", ""))

    st.divider()
    st.subheader("Timestamps")
    for ts in data.get("timestamps", []):
        t_time = ts.get("time", "")
        t_desc = ts.get("description", "")
        st.markdown(f"**{t_time}** - {t_desc}")

    st.divider()
    st.subheader("Flaws & Improvements")
    for flaw in data.get("flaws", []):
        issue = flaw.get("issue", "")
        why   = flaw.get("why_it_hurts", "")
        fix   = flaw.get("fix", "")
        st.markdown(f"**Issue:** {issue}\n\n**Why:** {why}\n\n**Fix:** {fix}")
'''

# Write app.py to disk
with open('app.py', 'w') as fh:
    fh.write(APP_CODE)
print('app.py written to disk.')

# Install localtunnel (no account or token needed, no tunnel conflicts)
subprocess.run(['npm', 'install', '-g', 'localtunnel'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Kill any leftover Streamlit on port 8501
subprocess.run(['fuser', '-k', '8501/tcp'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

# Start Streamlit
subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.enableCORS=false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(4)

# Start localtunnel and capture the public URL from its stdout
lt_proc = subprocess.Popen(
    ['lt', '--port', '8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.DEVNULL,
    text=True,
)

# Read the URL line from localtunnel output
public_url = ''
for line in lt_proc.stdout:
    if 'loca.lt' in line or 'localtunnel' in line.lower():
        public_url = line.strip().split()[-1]
        break

import requests as _req

if public_url:
    try:
        tunnel_ip = _req.get('https://ipv4.icanhazip.com', timeout=5).text.strip()
        print(f'\nStreamlit app live at: {public_url}')
        print(f'If asked for a password, enter: {tunnel_ip}')
    except Exception:
        print(f'\nStreamlit app live at: {public_url}')
        print('If asked for a password, visit https://ipv4.icanhazip.com to get your IP.')
else:
    print('Could not read tunnel URL. Check if localtunnel installed correctly.')

app.py written to disk.

Streamlit app live at: https://plain-dolls-make.loca.lt
If asked for a password, enter: 8.229.58.215


## Cell 12 - Zip and Download All Outputs
Compresses everything in `OUTPUT_DIR` (JSON, tags, timestamps, flaws,
MLflow runs) into a single archive and triggers a browser download.

In [23]:
import shutil
from google.colab import files  # type: ignore

# ── Archive path (shutil.make_archive appends .zip automatically) ─────────────
ZIP_BASE = '/content/yt_seo_outputs'
ZIP_FILE = ZIP_BASE + '.zip'

# ── Create the zip from the OUTPUT_DIR tree ───────────────────────────────────
shutil.make_archive(
    base_name=ZIP_BASE,
    format='zip',
    root_dir=OUTPUT_DIR,
)

print(f'Archive created: {ZIP_FILE}')

# ── Trigger browser download ──────────────────────────────────────────────────
files.download(ZIP_FILE)
print('Download triggered. Check your browser download bar.')


Archive created: /content/yt_seo_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered. Check your browser download bar.
